## Session 10, in class: Reiter's method

Computational Macro (HWS 2026), University of Mannheim. Thursday, 12.11.2026, 10:15 to 11:45.

This is the code behind the session 10 slides. The economy has households with CARA utility, persistent income
risk, and a constant probability of death (perpetual youth), and a firm that rents their savings as capital. It has
an aggregate version of five equations that is exact (a PRANK), so we know the answer before we start. Part 1 solves
that aggregate version with the code of week 9. Part 2 solves the steady state of the heterogeneous agent economy on
the grid, with the tools of weeks 5 to 8, and compares it with the analytical policy and distribution. Part 3 is
Reiter's method: the histogram and the marginal values on the grid become variables of one large system
$\mathbf{F}(\mathbf{x}_{t+1}, \mathbf{x}_t) = \mathbf{0}$, which we linearize and solve with Klein, as last week. Part 4
compares the impulse responses with those of the aggregate version, part 5 with the same economy without income
risk.

**How to work with this notebook**

- Run the cells from top to bottom (`Shift + Enter`). After a restart of the kernel, start again at the top:
  later cells use what earlier cells define.
- There are **8 gaps**, marked `___`. The comment line `# Gap k: ...` directly above a gap says what goes in.
  Every gap is one line, and most are one line of the slides.
- Cells without a gap run as they are. Read them anyway: most of them are functions from weeks 5 to 9, and the
  comments say where they changed.
- Each part ends with a **Check**: a number to compare with, from the slides. Do not move on before it fits.
- **Slides.** "Slide 14" and the like refer to the session 10 slides, by the number at the bottom of each slide.
  Keep them open next to the notebook.
- Part 3 takes a few seconds per cell: the system has 801 variables. The solution notebook is posted after the session.

**Plan**

| Part | Content | Gaps | Slides | Time |
| --- | --- | --- | --- | --- |
| 1 | The aggregate economy (PRANK): steady state, linearization, Klein | 1, 2 | 4 to 7 | 10 min |
| 2 | The steady state on the grid: EGM on marginal values, Young with newborns, against the analytical solution | 3, 4 | 8 to 11 | 25 min |
| 3 | Reiter's method: the system with 801 variables, its linearization, Klein | 5, 6 | 12 to 17 | 30 min |
| 4 | Impulse responses: Reiter against the PRANK | 7 | 18 | 10 min |
| 5 | With and without income risk, and experiments | 8 | 19 | 15 min |

**Notation, slides to code**

| Slides | Meaning | Code |
| --- | --- | --- |
| $a_i$, $e_m$ | asset and income grid points | `gri.a`, `gri.e` |
| $x = \frac{R}{\varphi} a + w (1 + e)$ | cash on hand: the annuity return on assets plus income | `x` |
| $v = \frac{R}{\varphi} \, u'(c)$ | the marginal value of assets on the grid, in logs | `lnv`, an $N_a \times N_e$ table |
| $a'(a_i, e_m)$, $c(a_i, e_m)$ | the savings and consumption policies on the grid | `a_prime`, `c` |
| $\boldsymbol{\lambda}$, $\boldsymbol{\lambda}^0$ | the histogram over $(a_i, e_m)$, and the one of the newborns | `lam`, `lam_born` |
| $\mathbf{Q}$ | the policy chain of week 6: survivors move from cell to cell | `Q` |
| $\mathbf{x}_t = (\boldsymbol{\lambda}_t, K_t, Z_t, \mathbf{v}_t)$ | the variables of Reiter's system, states first | `x`, a vector of length 801 |
| $\mathbf{A}$, $\mathbf{B}$, $\mathbf{G}$, $\mathbf{H}$ | as in week 9 | `A`, `B`, `G`, `H` |

**Toolbox: the few tools this notebook uses that are new**

| Tool | What it does | Example |
| --- | --- | --- |
| `[a; b; c]` | one long vector from several pieces (numbers or vectors) | the vector $\mathbf{x}_t$ |
| `x[1:N-1]`, `x[N+2:end]` | the first $N - 1$ entries, everything from entry $N + 2$ on | taking $\mathbf{x}_t$ apart |
| `reshape(v, na, ne)`, `vec(M)` | a vector as an $N_a \times N_e$ table and back, column by column | the histogram as a table |
| `schur(A, B)` on real matrices | the generalized Schur decomposition in real arithmetic, about four times faster than with `complex` | Klein with 801 variables |
| `cdf(Normal(), z)` from `Distributions` | the CDF $\Phi(z)$ of the standard normal distribution | the analytical distribution |

Everything else is from the earlier notebooks: `Base.@kwdef struct`, `rouwenhorst`, `sparse`, the dots `.+`, `.*`,
and the functions `jacobian`, `linearize`, `solve_klein`, `impulse_response` of week 9.

### 0. Packages

`LinearAlgebra`, `SparseArrays`, `Printf`, and `Statistics` ship with Julia, `Plots` and `QuantEcon` are in the
course environment. New this week: `Distributions`, for the CDF of the normal distribution. It is already on your
laptop as a part of `QuantEcon`, and the course environment now lists it. The first cell activates the course
environment in `Julia/` and installs every package listed there that is missing on your laptop.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # the course environment in Julia/
Pkg.instantiate()                        # installs every package it lists that is missing on your laptop

In [ ]:
using Plots
using Printf
using LinearAlgebra
using SparseArrays                                        # sparse matrices, for the policy chain
using Statistics
using QuantEcon                                           # rouwenhorst, as in week 5
using Distributions                                       # Normal and cdf, for the analytical distribution in part 2

### 1. The aggregate economy: a PRANK with five equations (gaps 1 and 2)

*Slides 4 to 7.*

**Households.** A continuum of households of mass one. Each period a household survives with probability
$\varphi$, and every household that dies is replaced by a newborn without assets (perpetual youth, slide 4). Utility is
CARA, $u(c) = -e^{-\gamma c} / \gamma$, there is no borrowing limit, and income is $w_t (1 + e_t)$ with
$e_{t+1} = \rho_e e_t + \varepsilon_{t+1}$. Savings are annuities: the assets of the dead are paid out to the
survivors, so that one unit saved pays $R_t / \varphi$ with $R_t = 1 + r_t$. The household's problem is the one of
problem set 2 with $\beta \varphi$ in place of $\beta$ and $R / \varphi$ in place of $R$, and its policy is affine:
$c = \mu x + \nu w e + f$ with cash on hand $x = \frac{R}{\varphi} a + w (1 + e)$ (slide 6).

**Firm and TFP.** $Y_t = Z_t K_t^\alpha$, labor is one, so $R_t = \alpha Z_t K_t^{\alpha - 1} + 1 - \delta$ and
$w_t = (1 - \alpha) Z_t K_t^\alpha$. Log TFP is an AR(1) with persistence $\rho_Z$ around $\ln \bar Z$.

**The aggregate version (slide 7).** Since $\mu_t$ is the same for every household, aggregate consumption
depends on aggregates only. Five equations in $\mathbf{x}_t = (K_t, Z_t, C_t, \mu_t, \nu_t)$, states first:

$$
\begin{aligned}
K_{t+1} &= Z_t K_t^\alpha + (1 - \delta) K_t - C_t && \text{resource constraint} \\
\ln Z_{t+1} - \ln \bar Z &= \rho_Z (\ln Z_t - \ln \bar Z) && \text{TFP} \\
C_{t+1} &= C_t + \tfrac{1}{\gamma} \ln (\beta R_{t+1}) - \tfrac{1 - \varphi}{\varphi} \mu_{t+1} R_{t+1} K_{t+1}
          + \tfrac{\gamma}{2} (\mu_{t+1} + \nu_{t+1})^2 w_{t+1}^2 \sigma_\varepsilon^2 && \text{aggregate Euler equation} \\
\mu_t &= \frac{\mu_{t+1} R_{t+1}}{\varphi + \mu_{t+1} R_{t+1}} && \text{the MPC} \\
\nu_t &= \rho_e \frac{\varphi}{R_{t+1}} (\mu_{t+1} + \nu_{t+1}) \frac{w_{t+1}}{w_t} && \text{the loading on } e
\end{aligned}
$$

with $\sigma_\varepsilon^2 = \sigma_e^2 (1 - \rho_e^2)$ the variance of the innovation. **Calibration:** $\beta$ and
$\bar Z$ are chosen so that the steady state has $r = 4$ percent and $Y = 1$. In the steady state the MPC is the
one of problem set 2 with $R / \varphi$ in place of $R$, and the aggregate Euler equation with $C_{t+1} = C_t$
gives $\beta$.

In [ ]:
Base.@kwdef struct EconomicParameters
    α::Float64 = 0.36         # capital share
    δ::Float64 = 0.1          # depreciation rate
    γ::Float64 = 4.0          # absolute risk aversion, CARA utility u(c) = -exp(-γ c) / γ
    φ::Float64 = 44 / 45      # survival probability: 45 periods of life on average
    ρ_e::Float64 = 0.9        # persistence of the income shock e
    σ_e::Float64 = 0.1        # unconditional standard deviation of e
    ρ_Z::Float64 = 0.9        # persistence of log TFP
    β::Float64 = 0.97         # discount factor, calibrated below
    Zbar::Float64 = 1.0       # steady state TFP, calibrated below
end

In [ ]:
function prank_steady_state(par; r = 0.04, Y = 1.0)
    # The steady state of the aggregate economy with the rate r and output Y (slide 7): the firm gives K, Zbar, w,
    # the household's coefficients μ and ν, and the aggregate Euler equation with C' = C gives β
    R = 1 + r
    K = par.α * Y / (r + par.δ)                                               # the firm: r = α Y / K - δ
    Zbar = Y / K^par.α
    w = (1 - par.α) * Y
    C = Y - par.δ * K                                                         # the resource constraint, K' = K
    # Gap 1: the MPC out of cash on hand: problem set 2's μ = 1 - 1/R, with the annuity return R / φ in place
    #        of R
    μ = ___
    ν = par.ρ_e * par.φ * μ / (R - par.ρ_e * par.φ)                           # the extra response to a persistent e
    sig_eps2 = par.σ_e^2 * (1 - par.ρ_e^2)                                    # variance of the innovation to e
    ln_βR = par.γ * ((1 - par.φ) / par.φ * μ * R * K - par.γ / 2 * (μ + ν)^2 * w^2 * sig_eps2)
    return (K = K, C = C, Z = Zbar, R = R, w = w, μ = μ, ν = ν, β = exp(ln_βR) / R, Y = Y)
end

In [ ]:
ss = prank_steady_state(EconomicParameters())
par = EconomicParameters(β = ss.β, Zbar = ss.Z)                              # the calibrated economy
for (key, val) in pairs(ss)
    @printf("%4s: %.6f\n", key, val)
end

**Check (slide 5):** $K = 2.5714$, $C = 0.7429$, $\bar Z = 0.7118$, $w = 0.64$, $\mu = 0.0598$, $\nu = 0.3291$, and
$\beta = 0.9747$. Note $\beta R = 1.014 > 1$: an infinitely lived household would let its consumption grow without
bound. Here the newborns, who own nothing, hold aggregate consumption down.

**The system and its linearization.** `F_prank` writes the five equations as residuals, with the variables in logs as
in week 9. The functions below it are the ones of week 9, unchanged except for one call in `solve_klein`: the
decomposition is done in real arithmetic (`schur(A, B)` without `complex`), which gives the same $\mathbf{G}$ and
$\mathbf{H}$ and is about four times faster. That does not matter for five variables, it does for 801 (slide 16).

In [ ]:
function prices(K, Z, par)
    # The firm's first-order conditions: the gross return R = 1 + r and the wage w
    R = par.α * Z * K^(par.α - 1) + 1 - par.δ
    w = (1 - par.α) * Z * K^par.α
    return R, w
end

In [ ]:
function F_prank(x_next, x, par)
    # The five equations of slide 7 as residuals. x = (ln K, ln Z, ln C, ln μ, ln ν) today, x_next tomorrow
    K, Z, C, μ, ν = exp.(x)
    K1, Z1, C1, μ1, ν1 = exp.(x_next)
    R, w = prices(K, Z, par)
    R1, w1 = prices(K1, Z1, par)
    sig_eps2 = par.σ_e^2 * (1 - par.ρ_e^2)
    return [
        log(K1) - log(Z * K^par.α + (1 - par.δ) * K - C),                     # resource constraint
        log(Z1 / par.Zbar) - par.ρ_Z * log(Z / par.Zbar),                     # TFP, without the shock
        # Gap 2: the term of the newborns in the aggregate Euler equation (slide 7): they own none of K'
        C1 - C - log(par.β * R1) / par.γ + ___ -
            par.γ / 2 * (μ1 + ν1)^2 * w1^2 * sig_eps2,                        # aggregate Euler equation
        log(μ) - log(μ1 * R1 / (par.φ + μ1 * R1)),                            # the MPC
        log(ν) - log(par.ρ_e * par.φ / R1 * (μ1 + ν1) * w1 / w),              # the loading on e
    ]
end

In [ ]:
# Check: all five residuals are zero at the steady state
xss_p = log.([ss.K, ss.Z, ss.C, ss.μ, ss.ν])
println("F at the steady state: ", F_prank(xss_p, xss_p, par))

In [ ]:
function jacobian(f, x0; h = 1e-6)
    # The Jacobian of f at x0 by central differences: column j is df / dx_j (week 9)
    n, m = length(x0), length(f(x0))
    J = zeros(m, n)
    for j in 1:n
        e = zeros(n)
        e[j] = h
        J[:, j] = (f(x0 .+ e) .- f(x0 .- e)) ./ (2h)
    end
    return J
end

function linearize(F, xss, par)
    # The matrices A and B of the linear system A E x' = B x at the steady state xss (week 9)
    A = jacobian(x_next -> F(x_next, xss, par), xss)
    B = -jacobian(x -> F(xss, x, par), xss)
    return A, B
end

In [ ]:
function solve_klein(A, B, n_states)
    # Klein (2000), as in week 9, with one change: schur(A, B) in real arithmetic, the same G and H, about four times faster
    Fs = schur(A, B)                                                          # A = Q S Z', B = Q T Z'
    stable = abs.(Fs.β) .< abs.(Fs.α)
    n_stable = sum(stable)
    if n_stable != n_states
        println("Klein: $n_stable stable eigenvalues for $n_states states")
    end
    ordschur!(Fs, stable)                                                     # the stable eigenvalues first
    Z11, Z21 = Fs.Z[1:n_states, 1:n_states], Fs.Z[n_states+1:end, 1:n_states]
    S11, T11 = Fs.S[1:n_states, 1:n_states], Fs.T[1:n_states, 1:n_states]
    G = Z21 / Z11                                                             # the controls on the states
    H = Z11 * (S11 \ T11) / Z11                                               # the law of motion of the states
    eigs = Fs.β ./ Fs.α                                                       # our λ_i = t_ii / s_ii
    return G, H, eigs
end

function impulse_response(G, H, n_states, T, shock)
    # The paths of states and controls for T periods after the shock vector hits the states in period 0 (week 9)
    s = zeros(T, n_states)
    s[1, :] = shock
    for t in 1:T-1
        s[t+1, :] = H * s[t, :]
    end
    return s, s * G'
end

In [ ]:
A_p, B_p = linearize(F_prank, xss_p, par)
G_p, H_p, eigs_p = solve_klein(A_p, B_p, 2)
println("eigenvalues: ", round.(sort(abs.(eigs_p)), digits = 4))
println("G (rows C, μ, ν on K, Z) ="); display(round.(G_p, digits = 4))
println("H ="); display(round.(H_p, digits = 4))
T = 60
s_p, c_p = impulse_response(G_p, H_p, 2, T, [0.0, 0.01])                     # a one percent TFP shock
@printf("C on impact %.3f percent, K peaks at %.3f percent in period %d\n", 100 * c_p[1, 1], 100 * maximum(s_p[:, 1]), argmax(s_p[:, 1]) - 1)

**Check (slide 7):** all five residuals are zero at the steady state, two stable eigenvalues for two states,
$0.900 = \rho_Z$ and $0.927$, the capital root, and three unstable ones, $1.052$, $1.129$, $1.187$. Consumption rises by
$0.552$ percent on impact, capital peaks at $1.027$ percent in period $11$. These are the numbers that 801 variables
have to reproduce in part 4.

### 2. The steady state on the grid (gaps 3 and 4)

*Slides 8 to 11.*

Now the same economy with the households on the grid, as in weeks 5 to 8: Rouwenhorst for $e$ with $N_e = 5$ points,
and $N_a = 80$ quadratically spaced asset points on $[-3, 50]$ (no borrowing limit, so the grid has to reach where
households go). Two things change against week 8 (slide 8).

**1. The EGM iterates on marginal values.** Week 5's `egm_step` took the consumption policy of tomorrow. Here it takes
the marginal value of assets on the grid, $v(a, e) = \frac{R}{\varphi} u'(c(a, e))$ (the envelope condition), in
logs. One step: the Euler equation $u'(c) = \beta \varphi \, \mathbb{E}[v' \mid e]$ gives consumption at every
$(a'_l, e_m)$, the budget constraint the cash on hand at which $a'_l$ is chosen, and the interpolation back to the
grid the savings policy. With CARA, $\ln v = \ln (R / \varphi) - \gamma c$: the log marginal value is consumption,
up to a constant. The reason to iterate on $v$: $v_{t+1}$ contains tomorrow's return, so that one step only needs
today's prices. That is what Reiter's system in part 3 uses.

**2. Newborns.** After the survivors have moved along the policy chain $\mathbf{Q}$ of week 6, the dead, a share
$1 - \varphi$ of the population, are replaced by newborns with $a = 0$ and $e$ drawn from its stationary distribution:
$\boldsymbol{\lambda}' = \varphi \mathbf{Q}^\top \boldsymbol{\lambda} + (1 - \varphi) \boldsymbol{\lambda}^0$.

In [ ]:
Base.@kwdef struct NumericalParameters
    na::Int = 80              # asset grid points N_a
    ne::Int = 5               # income grid points N_e
    amin::Float64 = -3.0      # lowest asset grid point: no borrowing limit, the grid has to reach where households go
    amax::Float64 = 50.0      # highest asset grid point
    crit_pol::Float64 = 1e-11 # the EGM stops when ln v moves by less: Reiter needs a tight steady state (slide 15)
    crit_dist::Float64 = 1e-13  # the distribution iteration stops when lam moves by less
    maxiter::Int = 100_000    # iteration cap, the safety net
end

struct Grids
    a::Vector{Float64}        # asset grid, N_a
    e::Vector{Float64}        # income grid, N_e
    Pi::Matrix{Float64}       # transition matrix of e, N_e x N_e
    lam_e::Vector{Float64}    # stationary distribution of e, N_e
end

function make_grids(par, mpar)
    # Rouwenhorst for e (week 5): rouwenhorst wants the standard deviation of the innovation, σ_e sqrt(1 - ρ_e^2),
    # and the quadratically spaced asset grid of week 6
    mc = rouwenhorst(mpar.ne, par.ρ_e, par.σ_e * sqrt(1 - par.ρ_e^2))
    s = collect(range(0.0, 1.0, length = mpar.na))
    a = mpar.amin .+ (mpar.amax - mpar.amin) .* s .^ 2
    return Grids(a, collect(mc.state_values), Matrix(mc.p), vec(stationary_distributions(mc)[1]))
end

In [ ]:
mpar = NumericalParameters()
gri = make_grids(par, mpar)
println("income states e: ", round.(gri.e, digits = 4), "  stationary distribution: ", round.(gri.lam_e, digits = 4),
        "  mean ", round(gri.lam_e' * gri.e, digits = 12))
@printf("asset grid: %d points on [%.1f, %.1f], spacing at 0: %.3f\n", mpar.na, gri.a[1], gri.a[end], diff(gri.a)[searchsortedfirst(gri.a, 0) - 1])

In [ ]:
function interp_extrap(x, xp, fp)
    # Linear interpolation of the points (xp, fp) at x, with linear extrapolation beyond the ends (week 9)
    out = similar(x)
    for (k, xq) in enumerate(x)
        i = clamp(searchsortedlast(xp, xq), 1, length(xp) - 1)               # the interval [xp[i], xp[i+1]] around xq
        w = (xq - xp[i]) / (xp[i+1] - xp[i])                                  # weight on the right end point, not clamped
        out[k] = (1 - w) * fp[i] + w * fp[i+1]
    end
    return out
end

In [ ]:
function egm_step(lnv_next, R, w, par, gri)
    # One step of the EGM on marginal values (slide 8): from ln v' on the grid (a'_l, e_n) to today's savings policy a_prime,
    # consumption c, and ln v, all N_a x N_e, at today's prices R and w
    Ev = par.φ .* exp.(lnv_next) * gri.Pi'                                    # E[v' | e_m] at every a'_l, the dead need no assets
    # Gap 3: (a) the Euler equation u'(c) = β E[v'] with u'(c) = exp(-γ c), solved for c
    ctilde = ___
    xtilde = ctilde .+ gri.a                                                  # (b) the cash on hand at which a'_l is chosen
    x = R / par.φ .* gri.a .+ w .* (1 .+ gri.e')                              # cash on hand on the grid
    a_prime = similar(x)
    for m in 1:length(gri.e)                                                  # (c) back to the grid, one income state at a time
        a_prime[:, m] = interp_extrap(x[:, m], xtilde[:, m], gri.a)
    end
    c = x .- a_prime
    # Gap 4: (d) the envelope condition v = (R / φ) u'(c), in logs
    lnv = ___
    return a_prime, c, lnv
end

In [ ]:
function solve_household(R, w, par, mpar, gri; lnv0 = nothing)
    # Iterate egm_step at constant prices until ln v stops moving (week 5). The guess: keep assets constant, a' = a
    if lnv0 === nothing
        x = R / par.φ .* gri.a .+ w .* (1 .+ gri.e')
        lnv0 = log(R / par.φ) .- par.γ .* (x .- gri.a)
    end
    lnv, a_prime, c, it = lnv0, similar(lnv0), similar(lnv0), 0
    while it < mpar.maxiter
        it += 1
        a_prime, c, lnv_new = egm_step(lnv, R, w, par, gri)
        dist = maximum(abs.(lnv_new .- lnv))
        lnv = lnv_new
        dist < mpar.crit_pol && break
    end
    return a_prime, c, lnv, it
end

**Young's method with newborns.** `lottery` and `transition_matrix` are the functions of week 8, with the income
grid now called `e`. The newborns are a histogram of mass one: assets $0$, put on the two grid points around $0$ by
the same lottery, and income from the stationary distribution of the chain. `forward` moves the histogram by one
period, `stationary_dist` iterates it until it stops moving (week 6).

In [ ]:
function lottery(a_prime, gri)
    # For every policy value a': the index l of the grid point just below it, and the weight w on the point above (week 6)
    na = length(gri.a)
    l = clamp.(searchsortedlast.(Ref(gri.a), a_prime), 1, na - 1)
    w = clamp.((a_prime .- gri.a[l]) ./ (gri.a[l .+ 1] .- gri.a[l]), 0.0, 1.0)
    return l, w
end

function transition_matrix(a_prime, gri)
    # The policy chain Q of week 6: Q[(i, m), (l, n)] = Pi[m, n] (1 - w_im) and Q[(i, m), (l + 1, n)] = Pi[m, n] w_im
    na, ne = length(gri.a), length(gri.e)
    idx = reshape(1:(na * ne), na, ne)                                        # idx[i, m] = i + (m - 1) N_a
    l, w = lottery(a_prime, gri)
    rows, cols, vals = Int[], Int[], Float64[]
    for m in 1:ne
        for n in 1:ne
            append!(rows, idx[:, m]); append!(cols, idx[l[:, m], n]); append!(vals, gri.Pi[m, n] .* (1 .- w[:, m]))
            append!(rows, idx[:, m]); append!(cols, idx[l[:, m] .+ 1, n]); append!(vals, gri.Pi[m, n] .* w[:, m])
        end
    end
    return sparse(rows, cols, vals, na * ne, na * ne)
end

In [ ]:
function newborns(gri)
    # The newborns as a histogram of mass one: a = 0 by the lottery, e from its stationary distribution
    l, w = lottery(zeros(1, 1), gri)
    lam_born = zeros(length(gri.a), length(gri.e))
    lam_born[l[1, 1], :] = (1 - w[1, 1]) .* gri.lam_e
    lam_born[l[1, 1] + 1, :] = w[1, 1] .* gri.lam_e
    return lam_born
end

function forward(lam, Q, lam_born, par)
    # One period of the histogram (slide 8): survivors move along the policy chain, the dead are replaced by newborns
    survivors = Q' * vec(lam)                                                 # week 6: lam' = Q' lam, as a vector
    # the dead, a share 1 - φ of all households (lam sums to one), come back as newborns
    lam_new = par.φ .* survivors .+ (1 - par.φ) * sum(lam) .* vec(lam_born)
    return reshape(lam_new, size(lam))
end

function stationary_dist(Q, lam_born, par, mpar; lam0 = nothing)
    # Iterate forward until the histogram stops moving (week 6), from the newborns or from lam0
    lam = lam0 === nothing ? copy(lam_born) : lam0
    it = 0
    while it < mpar.maxiter
        it += 1
        lam_new = forward(lam, Q, lam_born, par)
        dist = maximum(abs.(lam_new .- lam))
        lam = lam_new
        dist < mpar.crit_dist && break
    end
    return lam, it
end

In [ ]:
lam_born = newborns(gri)
println("newborns: mass ", sum(lam_born), "  mean assets ", sum(lam_born .* gri.a))

**The equilibrium.** At a rate $r$ the firm demands $K(r) = (\alpha \bar Z / (r + \delta))^{1 / (1 - \alpha)}$, which
also fixes the wage. The households supply what they save, $S(r) = \sum_{i, m} \lambda_{im} \, a'(a_i, e_m)$: today's
savings are tomorrow's capital. Supply increases in $r$, so we bisect on $S(r) - K(r)$ as in weeks 6 to 8, with warm
starts. The calibration has $r = 4$ percent in the aggregate economy, so that is where the bisection should land.

In [ ]:
mutable struct Aggregates
    S::Float64                # asset supply: mean savings of the households
    K::Float64                # capital demand of the firm
    R::Float64                # gross return 1 + r
    w::Float64                # wage
    a_prime::Matrix{Float64}  # savings policy a'(a, e), N_a x N_e
    c::Matrix{Float64}        # consumption policy
    lnv::Matrix{Float64}      # log marginal values
    lam::Matrix{Float64}      # stationary histogram
end

function aggregate(r, par, mpar, gri, lam_born; warm = nothing)
    # The household block at the rate r (weeks 7 and 8): prices, policy, histogram, supply S(r) and demand K(r)
    K = (par.α * par.Zbar / (r + par.δ))^(1 / (1 - par.α))                    # the firm's capital demand at r
    R, w = prices(K, par.Zbar, par)
    a_prime, c, lnv, _ = solve_household(R, w, par, mpar, gri; lnv0 = warm === nothing ? nothing : warm.lnv)
    lam, _ = stationary_dist(transition_matrix(a_prime, gri), lam_born, par, mpar; lam0 = warm === nothing ? nothing : warm.lam)
    return Aggregates(sum(lam .* a_prime), K, R, w, a_prime, c, lnv, lam)
end

function equilibrium(par, mpar, gri, lam_born, lo, hi; tol = 1e-12)
    # Bisection on the excess supply S(r) - K(r), which increases in r, with warm starts (week 8)
    warm, k = nothing, 0
    while hi - lo > tol
        mid = (lo + hi) / 2
        warm = aggregate(mid, par, mpar, gri, lam_born; warm = warm)
        lo, hi = warm.S < warm.K ? (mid, hi) : (lo, mid)
        k += 1
    end
    r = (lo + hi) / 2
    return r, aggregate(r, par, mpar, gri, lam_born; warm = warm), k
end

In [ ]:
t0 = time()
r_star, agg, k = equilibrium(par, mpar, gri, lam_born, 0.03, 0.05)
@printf("r* = %.5f percent after %d bisection steps, %.1f s\n", 100 * r_star, k, time() - t0)
@printf("supply S = %.6f, demand K = %.6f   (aggregate economy: K = %.6f)\n", agg.S, agg.K, ss.K)
@printf("C = sum lam c = %.6f   (aggregate economy: C = %.6f)\n", sum(agg.lam .* agg.c), ss.C)
@printf("mass %.12f, at the top of the grid %.1e, at the bottom %.1e\n", sum(agg.lam), sum(agg.lam[end, :]), sum(agg.lam[1, :]))

**Check (slide 10):** $r^* = 4.00012$ percent, $K = 2.57140$ against $2.57143$, $C = 0.74286$ against $0.74286$. The
grid economy has five income states instead of a normal distribution, which changes the precautionary saving a
little: that is the $10^{-6}$ in $r^*$. Almost no mass at the ends of the grid ($10^{-7}$ at the top). If $r^*$ is
far from $4$ percent, check gaps 3 and 4.

**Against the analytical solution.** The policy should be $c = \mu x + \nu w e + f$ with the coefficients of part 1
and $f = C - \mu X$, $X = R K + w$ (slide 6). For the distribution, the cash on hand of a household of age $k$ is
normal with a mean $m_k$ and a variance $V_k$ that follow simple recursions, so the stationary CDF is a mixture of
normals over ages, $F(x) = (1 - \varphi) \sum_{k \geq 0} \varphi^k \, \Phi\bigl( (x - m_k) / \sqrt{V_k} \bigr)$
(slide 11, the recursions are in the backup slides). `analytical_cdf` evaluates it.

In [ ]:
function analytical_cdf(x, par, ss; tol = 1e-13)
    # The stationary CDF of cash on hand (slide 11): a mixture over ages k of normals N(m_k, V_k), weights (1 - φ) φ^k
    Rφ, w, μ, ν = ss.R / par.φ, ss.w, ss.μ, ss.ν
    f = ss.C - μ * (ss.R * ss.K + w)                                          # the intercept of the policy
    κ = par.ρ_e - Rφ * ν                                                      # how much of e stays in x
    sig_eps2 = par.σ_e^2 * (1 - par.ρ_e^2)
    m, V, Cov = w, w^2 * par.σ_e^2, w * par.σ_e^2                             # age 0: x = w (1 + e), mean, variance, cov(x, e)
    d = w - Rφ * f                                                            # the mean grows by d per year of age
    F, weight = zeros(length(x)), 1 - par.φ
    while weight > tol
        F .+= weight .* cdf.(Normal(), (x .- m) ./ sqrt(V))                  # Φ from Distributions
        V, Cov = V + w^2 * κ^2 * par.σ_e^2 + w^2 * sig_eps2 + 2 * w * κ * Cov, par.ρ_e * Cov + w * (par.ρ_e * κ * par.σ_e^2 + sig_eps2)
        m += d
        weight *= par.φ
    end
    return F
end

In [ ]:
x_grid = agg.R / par.φ .* gri.a .+ agg.w .* (1 .+ gri.e')                    # cash on hand of every cell
X = ss.R * ss.K + ss.w
c_ana = ss.μ .* x_grid .+ ss.ν * ss.w .* gri.e' .+ (ss.C - ss.μ * X)
@printf("policy: largest |c - c_analytical| %.1e, weighted by the histogram %.1e\n", maximum(abs.(agg.c .- c_ana)), sum(agg.lam .* abs.(agg.c .- c_ana)))
slope = (agg.c[end, 3] - agg.c[1, 3]) / (x_grid[end, 3] - x_grid[1, 3])
@printf("slope of c in x: %.6f, μ = %.6f\n", slope, ss.μ)
order = sortperm(vec(x_grid))                                                 # the cells sorted by cash on hand
xs, cdf_num = vec(x_grid)[order], cumsum(vec(agg.lam)[order])
cdf_ana = analytical_cdf(xs, par, ss)
@printf("E[x] = %.5f against X = R K + w = %.5f\n", sum(agg.lam .* x_grid), X)
@printf("largest gap between the two CDFs: %.3f\n", maximum(abs.(cdf_num .- cdf_ana)))

In [ ]:
pa = plot(gri.a, agg.c[:, 3], lw = 2, label = "grid, e = 0", xlabel = "assets a", ylabel = "consumption", xlims = (-3, 20))
plot!(pa, gri.a, c_ana[:, 3], ls = :dash, color = :black, lw = 1, label = "analytical")
pb = plot(xs, cdf_num, lw = 2, label = "Young, N_a = $(mpar.na)", xlabel = "cash on hand x", ylabel = "CDF", xlims = (-1, 20), legend = :bottomright)
plot!(pb, xs, cdf_ana, ls = :dash, color = :black, lw = 1, label = "analytical")
plot(pa, pb, layout = (1, 2), size = (850, 320), bottom_margin = 4Plots.mm, left_margin = 4Plots.mm)

**Check (slides 10 and 11):** the policy is within $10^{-4}$ of the analytical one ($2 \cdot 10^{-5}$ weighted by the
histogram), the slope of $c$ in $x$ is $\mu = 0.05983$ to five digits, and $\mathbb{E}[x] = 3.31425$ against $X = 3.31429$: the lottery preserves means.
The CDFs differ by up to $0.07$ on this grid. That is the lottery: it puts the mass of a household on the two grid
points around its choice, which preserves the mean but adds variance. The mean, and with it every aggregate, is right
on any grid. On finer grids the gap shrinks (slide 11), experiment 2 at the end of the notebook lets you check it.

### 3. Reiter's method: the histogram and the marginal values as variables (gaps 5 and 6)

*Slides 12 to 17.*

Reiter's idea: on the grid, the distribution is a vector, and the steps that computed the steady state are
equations. The variables of period $t$ are, states first (slide 13),

| Block | Entries | Type | Deviation |
| --- | --- | --- | --- |
| histogram $\boldsymbol{\lambda}_t$, all bins but the last | $N_a N_e - 1 = 399$ | state | level |
| capital $K_t$ | $1$ | state | log |
| TFP $Z_t$ | $1$ | state | log |
| marginal values $\ln \mathbf{v}_t$ on the grid | $N_a N_e = 400$ | control | log |

and $\mathbf{F}(\mathbf{x}_{t+1}, \mathbf{x}_t)$ has four blocks (slide 12): Young's step forward, the asset market
$K_{t+1} = \sum \lambda_t \, a'_t$, TFP, and the EGM step backward,
$\ln \mathbf{v}_t = \ln (R_t / \varphi) - \gamma \mathbf{c}_t$, where $\mathbf{c}_t$ and $\mathbf{a}'_t$ come from
one `egm_step` on $\ln \mathbf{v}_{t+1}$ at today's prices (slide 8 draws the histogram as a vector and Young's step as a matrix product). In
the code the savings policy is called `a_prime`: in weeks 6 to 8 it was `G`, which here is Klein's $\mathbf{G}$. The
histogram is a state: it is given at the start of
the period. The marginal values are controls: they look forward, like consumption in week 9.

**Why one bin is missing (slide 15).** The histogram sums to one, so one bin is redundant: it is one minus the
others. Kept as a variable, it adds an eigenvalue of exactly one, which is neither stable nor explosive, and
Klein's count fails. **Levels for the histogram:** bins can be empty, and $\ln 0$ does not
exist. Levels are also the baseline of the BASEforHANK toolbox of Bayer, Born, and Luetticke. Everything else is in
logs, as in week 9.

In [ ]:
function unpack(x, gri)
    # The vector x of slide 13 as its four blocks: the histogram (with its last bin restored), K, Z, and ln v
    na, ne = length(gri.a), length(gri.e)
    N = na * ne
    # Gap 5: the last bin: the histogram sums to one
    lam = reshape([x[1:N-1]; ___], na, ne)
    lnv = reshape(x[N+2:end], na, ne)
    return lam, exp(x[N]), exp(x[N+1]), lnv
end

function pack(lam, K, Z, lnv)
    # The inverse of unpack: the four blocks as one vector, the histogram without its last bin
    N = length(lam)
    return [vec(lam)[1:N-1]; log(K); log(Z); vec(lnv)]
end

In [ ]:
function F_reiter(x_next, x, par, gri, lam_born)
    # Reiter's system (slides 12 and 14): zero along an equilibrium path without shocks
    lam, K, Z, lnv = unpack(x, gri)                                           # today
    lam1, K1, Z1, lnv1 = unpack(x_next, gri)                                  # tomorrow
    R, w = prices(K, Z, par)                                                  # today's prices
    a_prime, c, lnv_egm = egm_step(lnv1, R, w, par, gri)                            # one EGM step from tomorrow's v
    lam_fwd = forward(lam, transition_matrix(a_prime, gri), lam_born, par)          # one step of Young's method
    N = length(lam)
    return [vec(lam1 .- lam_fwd)[1:N-1];                                      # Young: all bins but the last
            # Gap 6: the asset market: tomorrow's capital is the mean of today's savings, in logs
            ___;
            log(Z1 / par.Zbar) - par.ρ_Z * log(Z / par.Zbar);                 # TFP, without the shock
            vec(lnv .- lnv_egm)]                                              # EGM: today's marginal values
end

In [ ]:
N = mpar.na * mpar.ne
xss = pack(agg.lam, agg.K, par.Zbar, agg.lnv)
# the number of states: the histogram without its last bin, capital, and TFP
n_states = N + 1
F_r(x_next, x, par) = F_reiter(x_next, x, par, gri, lam_born)                 # F with the grids filled in
println("$(length(xss)) variables, $n_states states, $(length(xss) - n_states) controls")
@printf("largest residual at the steady state: %.1e\n", maximum(abs.(F_r(xss, xss, par))))

**Check (slides 13 and 15):** $801$ variables, $401$ states, $400$ controls, and a residual below $10^{-10}$ at
the steady state. A residual of $10^{-5}$ or more means the steady state is not the one of $\mathbf{F}$: the
linearization would be around a point that is not a steady state. (At $r = 4$ percent instead of $r^*$ the residual
is $7 \cdot 10^{-5}$, try it with `aggregate(0.04, ...)`.)

**Linearize and solve.** The same two lines as in week 9. `linearize` evaluates $\mathbf{F}$ $4 \times 801$ times,
`solve_klein` decomposes two $801 \times 801$ matrices: a few seconds each (slide 16).

In [ ]:
t0 = time()
A, B = linearize(F_r, xss, par)
t_jac = time() - t0
t0 = time()
G, H, eigs = solve_klein(A, B, n_states)
t_qz = time() - t0
@printf("Jacobians: %.1f s, Klein: %.1f s\n", t_jac, t_qz)
mods = sort(replace(abs.(eigs), NaN => Inf))                                 # 0 / 0 counts as infinite
println("stable eigenvalues: $(sum(mods .< 1)) for $n_states states")
println("the five largest stable: ", round.(mods[mods .< 1][end-4:end], digits = 4))
println("the five smallest explosive: ", round.(mods[mods .> 1][1:5], digits = 4))

In [ ]:
finite = isfinite.(mods) .& (mods .> 0)
scatter(findall(finite), mods[finite], ms = 2, msw = 0, yscale = :log10, ylims = (1e-6, 1e3), label = "",
        xlabel = "eigenvalues, sorted by modulus", ylabel = "modulus", size = (700, 300),
        title = "$(sum(mods .< 1)) stable, $(sum(mods .> 1)) explosive (infinite ones not shown)", titlefontsize = 10)
hline!([1], color = :black, label = ""); vline!([n_states + 0.5], color = :gray, ls = :dot, label = "")

**Check (slides 16 and 17):** $401$ stable eigenvalues for $401$ states: the count fits, one bounded solution. The
largest stable ones sit just below $\varphi = 0.978$: the histogram forgets its past at the speed at which households
die. $\rho_Z = 0.9$ is among them. The smallest explosive one is $1.051$. On a laptop the Jacobians take a few
seconds, Klein between two and ten.

### 4. Impulse responses: Reiter against the PRANK (gap 7)

*Slide 18.*

A one percent TFP shock: the state vector starts at zero everywhere except for log TFP, and $\mathbf{H}$ and
$\mathbf{G}$ do the rest, exactly as in week 9. Capital and TFP are states, output follows from them,
$\hat Y_t = \hat Z_t + \alpha \hat K_t$. Consumption we compute twice: from the resource constraint,
$C_t = Y_t + (1 - \delta) K_t - K_{t+1}$, and from the households, $C_t = \sum \lambda_t c_t$, with the change of the
policy read off the marginal values, $\ln v = \ln (R / \varphi) - \gamma c$ gives
$dc = (d \ln R - d \ln v) / \gamma$. If the goods market clears, the two agree.

In [ ]:
s0 = zeros(n_states)
s0[N+1] = 0.01                                                                # log TFP is the last state
s_h, c_h = impulse_response(G, H, n_states, T, s0)                            # states and controls, T x 401 and T x 400
K_h, Z_h = s_h[:, N], s_h[:, N+1]                                             # log deviations of K and Z
Y_h = Z_h .+ par.α .* K_h
K, Y = agg.K, par.Zbar * agg.K^par.α
C = sum(agg.lam .* agg.c)
C_rc = (Y .* Y_h[1:end-1] .+ (1 - par.δ) * K .* K_h[1:end-1] .- K .* K_h[2:end]) ./ C;  # from the resource constraint, T - 1 periods

In [ ]:
function consumption_from_households(s_h, c_h, agg, par, gri)
    # C_t = sum lam_t c_t, linearized: d lam times c plus lam times dc, with dc from the marginal values
    na, ne = length(gri.a), length(gri.e)
    N = na * ne
    dlnR = (agg.R - 1 + par.δ) / agg.R .* (s_h[:, N+1] .+ (par.α - 1) .* s_h[:, N])   # d ln R from d ln Z and d ln K
    dC = zeros(size(s_h, 1))
    for t in 1:size(s_h, 1)
        dlam = reshape([s_h[t, 1:N-1]; -sum(s_h[t, 1:N-1])], na, ne)          # the change of the histogram, sums to zero
        dlnv = reshape(c_h[t, :], na, ne)                                     # the change of ln v on the grid
        # Gap 7: the change of consumption on the grid, from ln v = ln(R / φ) - γ c
        dc = ___
        dC[t] = sum(dlam .* agg.c) + sum(agg.lam .* dc)
    end
    return dC ./ sum(agg.lam .* agg.c)
end

In [ ]:
C_hh = consumption_from_households(s_h, c_h, agg, par, gri)
@printf("largest gap, capital, Reiter against the PRANK: %.1e\n", maximum(abs.(K_h .- s_p[:, 1])))
@printf("largest gap, consumption from the households against the resource constraint: %.1e\n", maximum(abs.(C_hh[1:end-1] .- C_rc)))
@printf("C on impact %.3f percent (PRANK %.3f), K peaks at %.3f percent in period %d\n", 100 * C_hh[1], 100 * c_p[1, 1], 100 * maximum(K_h), argmax(K_h) - 1)

In [ ]:
panels = []
for (name, ha, pr) in (("capital", K_h, s_p[:, 1]), ("consumption", C_hh, c_p[:, 1]), ("output", Y_h, s_p[:, 2] .+ par.α .* s_p[:, 1]))
    p = plot(0:T-1, 100 .* ha, lw = 2, label = "Reiter, 801 variables", title = name, titlefontsize = 10, xlabel = "periods after the shock")
    scatter!(p, 0:T-1, 100 .* pr, ms = 2, color = :black, label = "PRANK, 5 equations")
    hline!(p, [0], color = :black, lw = 0.5, label = "")
    push!(panels, p)
end
plot(panels..., layout = (1, 3), size = (950, 300), link = :y, bottom_margin = 4Plots.mm, left_margin = 4Plots.mm)

**Check (slide 18):** capital within $1.5 \cdot 10^{-5}$ of the PRANK at every horizon (in log points, at a peak of
$0.0103$), consumption on impact $0.552$ percent in both, capital peaks at $1.026$ percent in period $11$. The two
consumption series agree to $10^{-8}$: the goods market clears, as Walras's law says it must. 801 variables
and five equations give the same aggregate dynamics, because the households' MPC is the same for all of them.

### 5. With and without income risk (gap 8)

*Slide 19.*

Does the heterogeneity from income risk matter for the aggregates? The clean comparison keeps everything else: the
same newborns ($\varphi = 44/45$), the same utility, the same steady state, but no income risk, $\sigma_e = 0$.
Households still differ, by age and therefore by wealth, but no longer by luck. In the five equations of slide 7 only
the precautionary term changes: it is zero. The steady state keeps $r = 4$ percent and $Y = 1$, so `prank_steady_state`
gives the same $K$, $C$, and $\bar Z$ and a new $\beta$: without risk there is no precautionary saving, and a higher
$\beta$ has to hold the capital stock at the same level. The code is the one of part 1, only the parameters differ.

In [ ]:
# Gap 8: the same economy without income risk
ss_nr = prank_steady_state(EconomicParameters(___))
par_nr = EconomicParameters(β = ss_nr.β, Zbar = ss_nr.Z, σ_e = 0.0)
@printf("β without income risk %.5f, with income risk %.5f; K %.4f and %.4f\n", par_nr.β, par.β, ss_nr.K, ss.K)
xss_nr = log.([ss_nr.K, ss_nr.Z, ss_nr.C, ss_nr.μ, ss_nr.ν])
A_nr, B_nr = linearize(F_prank, xss_nr, par_nr)
G_nr, H_nr, eigs_nr = solve_klein(A_nr, B_nr, 2)
s_nr, c_nr = impulse_response(G_nr, H_nr, 2, T, [0.0, 0.01])
println("eigenvalues: ", round.(sort(abs.(eigs_nr)), digits = 4))
@printf("K peaks at %.3f percent (with income risk %.3f), C on impact %.3f percent (%.3f)\n", 100 * maximum(s_nr[:, 1]), 100 * maximum(K_h), 100 * c_nr[1, 1], 100 * C_hh[1])

In [ ]:
pa = plot(0:T-1, 100 .* K_h, lw = 2, label = "with income risk (Reiter)", title = "capital", titlefontsize = 10, xlabel = "periods after the shock", ylabel = "percent deviation")
plot!(pa, 0:T-1, 100 .* s_nr[:, 1], lw = 2, ls = :dash, label = "without income risk")
pb = plot(0:T-1, 100 .* C_hh, lw = 2, label = "", title = "consumption", titlefontsize = 10, xlabel = "periods after the shock")
plot!(pb, 0:T-1, 100 .* c_nr[:, 1], lw = 2, ls = :dash, label = "")
plot(pa, pb, layout = (1, 2), size = (800, 300), bottom_margin = 4Plots.mm, left_margin = 4Plots.mm)

**Check (slide 19):** $\beta = 0.97563$ without income risk against $0.97471$ with it, the same $K = 2.5714$.
Capital peaks at $1.025$ percent without income risk against $1.026$ with it, consumption rises by $0.553$ percent on
impact against $0.552$. The two lines lie on top of each other. With CARA utility every household has the same MPC
$\mu$, so who is rich and who is poor does not matter for aggregate consumption. Income risk enters the aggregate
only through the precautionary term, which is a few $10^{-4}$ of consumption and moves little after the shock.

**Experiments.**

1. **More income risk.** Set $\sigma_e = 0.3$ (three times the baseline) in the aggregate economy of part 1 and
   compare with the economy without risk. Does the gap grow? Then redo parts 2 to 4 on the grid: what do you have
   to change in `NumericalParameters` so that no mass reaches the ends of the asset grid?
2. **The grid: precision against cost.** Solve parts 2 and 3 with $N_a = 40$, $120$, and $200$
   (`NumericalParameters(na=...)`). In part 2, record the largest gap between the two CDFs and the standard deviation
   of cash on hand: how fast does the distribution approach the analytical one (slide 11)? In part 3, time the
   Jacobians and Klein: how does the time scale with $n$? Do the impulse responses of capital change? (They should
   not by much: the aggregates only need the mean of the distribution, which the lottery gets right on any grid.)
   Part 2 alone also runs with $N_a = 1000$; part 3 would then have $10{,}001$ variables.
3. **Real against complex.** Put `output="complex"` (Julia: `schur(complex(A), complex(B))`) back into
   `solve_klein`, as in week 9, and compare the time and $\mathbf{G}$.